In [9]:

import os
import sys
from google import genai
from datetime import datetime
from typing import List, Dict, Any
from PIL import Image
import requests
import json
from google.genai import types # This is used to access the GenerateContentConfig class for setting system instructions and other configurations.
from pydantic import BaseModel, Field
from typing import Literal


In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
SCRIPT_FILE    = "../scripts/Shrek.pdf"       # <-- path to your script (.txt, .pdf, .docx)
OUTPUT_FILE    = "../parsed_scripts/parsed_script.json"  # <-- desired output path
MODEL          = "gpt-4o-mini"         # options: gpt-4o-mini, gpt-4o, gpt-3.5-turbo

### Test Request

In [3]:
gemini_client = genai.Client(api_key=GOOGLE_API_KEY, http_options={'timeout': 100_000})
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=["You are a helpful assistant.", "Can you print 10 random words in a json format?"], # Contents = system + user messages
    config=types.GenerateContentConfig( # Pydantics parameters setter for controlling the response format and behavior of the model.
                                        response_mime_type="application/json",
)
)
print(response.text)

{
  "words": [
    "ephemeral",
    "serendipity",
    "ubiquitous",
    "mellifluous",
    "resplendent",
    "quixotic",
    "cacophony",
    "halcyon",
    "luminescent",
    "petrichor"
  ]
}


### Check Models available through gemini client

In [4]:
for model in gemini_client.models.list():
    if 'generateContent' in model.supported_actions:
        print(f"Model Name: {model.name}")

Model Name: models/gemini-2.5-flash
Model Name: models/gemini-2.5-pro
Model Name: models/gemini-2.0-flash
Model Name: models/gemini-2.0-flash-001
Model Name: models/gemini-2.0-flash-lite-001
Model Name: models/gemini-2.0-flash-lite
Model Name: models/gemini-2.5-flash-preview-tts
Model Name: models/gemini-2.5-pro-preview-tts
Model Name: models/gemma-3-1b-it
Model Name: models/gemma-3-4b-it
Model Name: models/gemma-3-12b-it
Model Name: models/gemma-3-27b-it
Model Name: models/gemma-3n-e4b-it
Model Name: models/gemma-3n-e2b-it
Model Name: models/gemini-flash-latest
Model Name: models/gemini-flash-lite-latest
Model Name: models/gemini-pro-latest
Model Name: models/gemini-2.5-flash-lite
Model Name: models/gemini-2.5-flash-image
Model Name: models/gemini-2.5-flash-lite-preview-09-2025
Model Name: models/gemini-3-pro-preview
Model Name: models/gemini-3-flash-preview
Model Name: models/gemini-3.1-pro-preview
Model Name: models/gemini-3.1-pro-preview-customtools
Model Name: models/gemini-3.1-fl

### Pydantic model for the expected output format of the parsed script lines

In [10]:
class ScriptLine(BaseModel):
    character: str = Field(description="The name of the character involved.")
    line: str = Field(description="The spoken dialogue. Leave empty if the action is silent.")
    action_type: Literal["speech", "sound", "act"] = Field(description="The category: 'speech' for talking, 'sound' for audio, 'act' for movement/camera.")
    action_description: str = Field(description="The actual physical description or camera direction (e.g., 'CLOSE ON: book', 'running away').")
    page_number: int

### Upload the script file to Gemini's file storage

In [13]:
# Disclaimer: The following prompt; without the 100 line limit, was used with Gemini 2.5 Flashlite, to prevent API cost and maintain testing of free, 
# it can be tested here and completed in AI Studio for free: https://aistudio.google.com/models/gemini-2.5-flash-lite 
# example of the output given when running this code is given below.

file = gemini_client.files.upload(file=SCRIPT_FILE)


prompt = (
    "Extract the first 5 character lines from this script. "
    "Include the character name, the dialogue, the action type, and the page number."
)

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite", # Low cost model, good for parsing and extracting information from documents. 
    contents=[
                file,
                prompt,
             ],
    config=types.GenerateContentConfig( # Pydantics parameters setter for controlling the response format and behavior of the model.
                                        temperature=0.0,
                                        response_mime_type="application/json", # Model's response is in JSON format
                                        system_instruction=( 
                                                           "You are a script analyst. For every entry:\n"
                                                           "1. Identify the 'character'.\n"
                                                           "2. If they are speaking, put the text in 'line' and set 'action_type' to 'speech'.\n"
                                                           "3. If there is a camera direction or movement (like 'CLOSE ON'), put that text in 'action_description' and set 'action_type' to 'act'.\n"
                                                           "4. NEVER put camera directions like 'CLOSE ON' inside the 'line' field.\n"
                                                           "5. If there is a sound effect (like 'SFX: thunder'), put that text in 'action_description' and set 'action_type' to 'sound'.\n"
                                                           "6. If there is no dialogue, leave 'line' empty.\n"
                                        ),
                                        response_schema={"type": "array", #
                                                         "items": ScriptLine.model_json_schema() # Pydantics mold for the output format
                                                         },
                                      )
)

print(response.text)



[
  {
    "character": "SHREK",
    "line": "Once upon a time there was a lovely princess.",
    "action_type": "speech",
    "action_description": "Narrator reading the book",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "But she had an enchantment upon her of a fearful sort which could only be broken by love's first kiss.",
    "action_type": "speech",
    "action_description": "cont'd",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "She was locked away in a castle guarded by a terrible, fire breathing dragon.",
    "action_type": "speech",
    "action_description": "cont'd",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "Many brave knights had attempted to free her from this dreadful prison, but none prevailed.",
    "action_type": "speech",
    "action_description": "cont'd",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "She waited in the Dragon's Keep, in the highest room of the tallest tower

In [12]:
# Parse the response JSON and save to output file
parsed_lines = json.loads(response.text)

with open(OUTPUT_FILE, "w") as f:
    json.dump(parsed_lines, f, indent=2)

print(f"Saved {len(parsed_lines)} script lines to {OUTPUT_FILE}")

Saved 1 script lines to ../parsed_scripts/parsed_script.json


### Cleanup (deletes the file from the cloud immediately)

In [ ]:
gemini_client.files.delete(name=file.name)  